# Lab 01 — Regression & Classification

## Business Context

You work as a data analyst at **ShopSmart**, a mid-sized e-commerce company. The marketing team has two urgent questions:

1. **How much will a customer spend** in the next 30 days? *(Regression)*
2. **Will a customer churn** (stop buying) in the next 90 days? *(Classification)*

Your job is to build predictive models to answer both questions.

---

## Learning Objectives

By the end of this lab you will be able to:
- Distinguish regression from classification problems
- Prepare data for ML (train/test split, feature scaling)
- Train and evaluate a regression model (Linear Regression)
- Train and evaluate a classification model (Logistic Regression, Decision Tree)
- Interpret key evaluation metrics (MAE, RMSE, Accuracy, F1, Confusion Matrix)

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

np.random.seed(42)
sns.set_theme(style='whitegrid')
print('Libraries loaded successfully!')

---

## Part 1 — Regression: Predicting Customer Spend

### 1.1 Generate the Dataset

We will simulate a customer dataset with the following features:

| Feature | Description |
|---|---|
| `days_since_last_purchase` | Days since the customer last bought something |
| `total_orders` | Total number of orders ever placed |
| `avg_order_value` | Average value (€) of past orders |
| `email_open_rate` | Fraction of marketing emails opened (0–1) |
| `spend_next_30d` | **Target** — spend in the next 30 days (€) |

In [ ]:
n = 500

days_since_last = np.random.exponential(scale=30, size=n).clip(1, 180).astype(int)
total_orders    = np.random.poisson(lam=8, size=n).clip(1, 50)
avg_order_value = np.random.normal(loc=75, scale=30, size=n).clip(10, 300)
email_open_rate = np.random.beta(a=2, b=5, size=n)

# Target: spend is driven by recency, order history, and average value
spend = (
    avg_order_value * 0.6
    + total_orders  * 2.5
    - days_since_last * 0.4
    + email_open_rate * 40
    + np.random.normal(0, 15, size=n)
).clip(0)

df_reg = pd.DataFrame({
    'days_since_last_purchase': days_since_last,
    'total_orders': total_orders,
    'avg_order_value': avg_order_value.round(2),
    'email_open_rate': email_open_rate.round(3),
    'spend_next_30d': spend.round(2)
})

df_reg.head()

### 1.2 Exploratory Data Analysis (EDA)

In [ ]:
df_reg.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features = ['days_since_last_purchase', 'total_orders', 'avg_order_value', 'email_open_rate']

for ax, feat in zip(axes.flat, features):
    ax.scatter(df_reg[feat], df_reg['spend_next_30d'], alpha=0.4, s=15)
    ax.set_xlabel(feat)
    ax.set_ylabel('spend_next_30d (€)')
    ax.set_title(f'{feat} vs Spend')

plt.tight_layout()
plt.show()

### 1.3 Prepare Data — Train / Test Split

In [ ]:
X_reg = df_reg.drop(columns='spend_next_30d')
y_reg = df_reg['spend_next_30d']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f'Train size: {len(X_train_r)} | Test size: {len(X_test_r)}')

### 1.4 Train a Linear Regression Model

In [ ]:
scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled  = scaler_r.transform(X_test_r)

lr = LinearRegression()
lr.fit(X_train_r_scaled, y_train_r)

y_pred_r = lr.predict(X_test_r_scaled)
print('Model trained!')

### 1.5 Evaluate the Regression Model

Key metrics:
- **MAE** (Mean Absolute Error) — average error in the same unit as the target (€)
- **RMSE** (Root Mean Squared Error) — penalises large errors more heavily
- **R²** — proportion of variance explained (1 = perfect, 0 = no better than predicting the mean)

In [ ]:
mae  = mean_absolute_error(y_test_r, y_pred_r)
rmse = mean_squared_error(y_test_r, y_pred_r, squared=False)
r2   = r2_score(y_test_r, y_pred_r)

print(f'MAE:  €{mae:.2f}')
print(f'RMSE: €{rmse:.2f}')
print(f'R²:    {r2:.3f}')

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test_r, y_pred_r, alpha=0.5, s=20)
plt.plot([y_test_r.min(), y_test_r.max()],
         [y_test_r.min(), y_test_r.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Spend (€)')
plt.ylabel('Predicted Spend (€)')
plt.title('Actual vs Predicted — Linear Regression')
plt.legend()
plt.tight_layout()
plt.show()

### 1.6 Feature Importance (Coefficients)

In [ ]:
coef_df = pd.DataFrame({
    'feature': X_reg.columns,
    'coefficient': lr.coef_
}).sort_values('coefficient', key=abs, ascending=False)

sns.barplot(data=coef_df, x='coefficient', y='feature', palette='coolwarm')
plt.title('Linear Regression Coefficients (scaled features)')
plt.tight_layout()
plt.show()

> **Discussion:** Which feature has the strongest influence on predicted spend? Does this match your business intuition?

---

## Part 2 — Classification: Will This Customer Churn?

### 2.1 Generate the Churn Dataset

In [ ]:
n = 600

tenure_months    = np.random.randint(1, 60, size=n)
monthly_spend    = np.random.normal(80, 35, size=n).clip(5, 300)
support_tickets  = np.random.poisson(1.5, size=n)
days_inactive    = np.random.exponential(20, size=n).clip(0, 120).astype(int)
has_subscription = np.random.binomial(1, 0.4, size=n)

# Churn probability influenced by inactivity, tickets, and lack of subscription
churn_prob = (
    0.03 * days_inactive
    + 0.08 * support_tickets
    - 0.4  * has_subscription
    - 0.01 * tenure_months
    + 0.5
)
churn_prob = 1 / (1 + np.exp(-churn_prob))  # sigmoid
churn = (np.random.rand(n) < churn_prob).astype(int)

df_clf = pd.DataFrame({
    'tenure_months': tenure_months,
    'monthly_spend': monthly_spend.round(2),
    'support_tickets': support_tickets,
    'days_inactive': days_inactive,
    'has_subscription': has_subscription,
    'churned': churn
})

print(df_clf['churned'].value_counts())
df_clf.head()

### 2.2 Quick EDA — Class Balance

In [ ]:
ax = df_clf['churned'].value_counts().rename({0: 'Retained', 1: 'Churned'}).plot(
    kind='bar', color=['steelblue', 'tomato'], edgecolor='white'
)
ax.set_title('Class Distribution')
ax.set_xlabel('')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

### 2.3 Prepare Data

In [ ]:
X_clf = df_clf.drop(columns='churned')
y_clf = df_clf['churned']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled  = scaler_c.transform(X_test_c)

### 2.4 Train — Logistic Regression

In [ ]:
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_c_scaled, y_train_c)

y_pred_log = log_reg.predict(X_test_c_scaled)

print('=== Logistic Regression ===')
print(classification_report(y_test_c, y_pred_log, target_names=['Retained', 'Churned']))

### 2.5 Train — Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train_c, y_train_c)  # Trees don't need scaling

y_pred_dt = dt.predict(X_test_c)

print('=== Decision Tree ===')
print(classification_report(y_test_c, y_pred_dt, target_names=['Retained', 'Churned']))

### 2.6 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, preds, title in zip(axes,
                             [y_pred_log, y_pred_dt],
                             ['Logistic Regression', 'Decision Tree']):
    cm = confusion_matrix(y_test_c, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Retained', 'Churned'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(title)

plt.tight_layout()
plt.show()

### 2.7 Visualise the Decision Tree

In [ ]:
plt.figure(figsize=(18, 6))
plot_tree(
    dt,
    feature_names=X_clf.columns,
    class_names=['Retained', 'Churned'],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title('Decision Tree — Churn Prediction')
plt.tight_layout()
plt.show()

---

## Your Turn! 🧪

**Exercise A — Regression:**
Try replacing `LinearRegression` with `sklearn.ensemble.RandomForestRegressor`. Does the R² improve?

**Exercise B — Classification:**
Change `max_depth` in the Decision Tree from 4 to 2 and then to 8. How do the results change? What does this tell you about overfitting?

**Exercise C — Think like a business analyst:**
Looking at the confusion matrix for churn prediction — which type of error is more costly for ShopSmart: a false positive (predicting churn when the customer stays) or a false negative (missing a real churner)? How would you adjust the model threshold?

In [ ]:
# Your code here


---

## Summary

| | Regression | Classification |
|---|---|---|
| **Output** | Continuous number | Discrete label |
| **Example** | Predicted spend (€) | Churned: Yes / No |
| **Key metrics** | MAE, RMSE, R² | Accuracy, Precision, Recall, F1 |
| **Model used** | Linear Regression | Logistic Regression, Decision Tree |

**Next session:** We'll move to unsupervised learning — clustering customers and detecting anomalous transactions.